In [13]:
# --  import libraries
import pandas as pd
import numpy as np
import pickle
import re
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# In this notebook we will be predicting the ACC using the Models we made

In [14]:
# read in data
matches_reference = pd.read_csv('matches_updated (1).csv'); display(matches_reference); print()
inference_pipeline_db = pd.read_csv('inference_pipeline_db.csv'); display(inference_pipeline_db); print()
teams = pd.read_csv('teams.xls'); display(teams); print()
duals = pd.read_csv('dual_meets.xls'); display(duals); print()
wrestlers = pd.read_csv('wrestlers_updated.csv'); display(wrestlers); print()
#

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
0,280,141,11/1/2025,267.0,Raymond Adams,160.0,825.0,John Hildebrandt,98.0,False,LDEC,LDEC 3 - 2,JR,Duke,JR,Sacred Heart
1,283,197,11/1/2025,878.0,Kael Bennie,46.0,263.0,Angelo Posada,20.0,False,LDEC,LDEC 5 - 4,SO,Utah Valley,FR,Stanford
2,283,285,11/1/2025,879.0,Jack Forbes,62.0,429.0,Jackson Mankowski,182.0,True,WDEC,WDEC 10 - 3,SR,Utah Valley,SO,Stanford
3,284,125,11/1/2025,784.0,Koda Holeman,40.0,530.0,Jacob Macatangay,69.0,True,WMD,WMD 20 - 8,JR,Cal Poly,JR,Purdue
4,284,133,11/1/2025,785.0,Anthony Lucio,245.0,226.0,Blake Boarman,57.0,False,LDEC,LDEC 8 - 2,RSFR,Cal Poly,SR,Purdue
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5835,547,174,2/23/2026,399.0,Joshua Roe,237.0,814.0,Beau Lewis,158.0,False,LDEC,LDEC 5 - 3,SO,Presbyterian,FR,VMI
5836,547,184,2/23/2026,400.0,Reed Douglass,72.0,815.0,Andrew Barford,162.0,True,WTF,WTF5 22 - 5 6:45,JR,Presbyterian,FR,VMI
5837,547,197,2/23/2026,911.0,Toler Hornick,207.0,816.0,Toby Schoffstall,74.0,False,LTF,LTF5 15 - 0 1:05,SO,Presbyterian,JR,VMI
5838,547,149,2/23/2026,398.0,Rey Ortiz,245.0,811.0,Patrick Jordon,77.0,False,LFALL,LFALL 4:22,SO,Presbyterian,JR,VMI


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff
0,William Morrow,Bloomsburg,206,75,[157],22,3,19,13.6,1,4.5,1,0,0,2,113.41,2.68,-2.77,5.20
1,Mason Rebuck,Bloomsburg,53,75,[285],20,15,5,75.0,10,50.0,2,2,6,5,127.85,8.17,3.92,8.62
2,Noah Mulvaney,Bucknell,32,36,[165],20,17,3,85.0,7,35.0,3,3,1,10,93.00,9.22,5.33,6.26
3,Jordan Soriano,Drexel,60,44,[141],20,14,6,70.0,7,35.0,0,2,5,7,112.05,7.31,2.23,6.22
4,Jesse Mendez,Ohio State,1,2,[141],19,19,0,100.0,17,89.5,9,3,5,2,57.37,16.64,12.36,4.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,Alaa Aly,Edinboro,208,47,[184],1,0,1,0.0,0,0.0,0,0,0,0,40.00,5.00,-15.00,0.00
1502,Cole Tolley,West Virginia,114,17,[197],1,0,1,0.0,0,0.0,0,0,0,0,63.00,6.00,-15.00,0.00
1503,Winston McBride,Wyoming,182,21,[285],1,1,0,100.0,0,0.0,0,0,0,1,240.00,1.00,1.00,0.00
1504,Aiden Robertson,Air Force,222,52,[174],1,0,1,0.0,0,0.0,0,0,0,0,76.00,0.00,-5.00,0.00


,team_id,name
0,1,Northern Iowa
1,2,South Dakota State
2,3,Drexel
3,4,Northern Illinois
4,5,Central Michigan
...,...,...
73,74,Indiana
74,75,Army West Point
75,76,Wyoming
76,77,Bellarmine


,dual_id,team1_id,team1_rank,team1_score,team2_id,team2_rank,team2_score,event_date
0,1,1,14,20,2,20,14,01/10/26
1,2,1,14,22,3,42,16,01/10/26
2,3,3,42,21,4,45,12,01/10/26
3,4,2,20,28,5,50,10,01/10/26
4,5,5,50,23,6,55,15,01/10/26
...,...,...,...,...,...,...,...,...
579,580,50,26,22,33,35,18,02/19/26
580,581,63,47,28,8,75,10,02/19/26
581,582,44,39,37,34,57,3,02/19/26
582,583,54,42,32,60,65,14,02/19/26


,wrestler_id,name,Class,Team
0,1,Bowen Downey,SO,Northern Iowa
1,2,Julian Farber,SR,Northern Iowa
2,3,Max Brady,FR,Northern Iowa
3,4,Ethan Basile,SR,Northern Iowa
4,5,Cael Rahnavardi,SR,Northern Iowa
...,...,...,...,...
1507,38,Landen Johnson,SO,Northern Illinois
1508,168,Nathan Taylor,SR,Lehigh
1509,897,James Conway,SR,Missouri
1510,2000,James Conway,SR,Franklin & Marshall


In [15]:
# -- Join class to inference_pipeline_db
inference_pipeline_db = inference_pipeline_db.merge(wrestlers[["name", "Team", "Class"]], left_on=['wrestler_name', 'team_name'], right_on = ['name', 'Team'])
inference_pipeline_db = inference_pipeline_db.drop(columns=['name', 'Team'])
inference_pipeline_db

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
0,William Morrow,Bloomsburg,206,75,[157],22,3,19,13.6,1,4.5,1,0,0,2,113.41,2.68,-2.77,5.20,SR
1,Mason Rebuck,Bloomsburg,53,75,[285],20,15,5,75.0,10,50.0,2,2,6,5,127.85,8.17,3.92,8.62,SO
2,Noah Mulvaney,Bucknell,32,36,[165],20,17,3,85.0,7,35.0,3,3,1,10,93.00,9.22,5.33,6.26,JR
3,Jordan Soriano,Drexel,60,44,[141],20,14,6,70.0,7,35.0,0,2,5,7,112.05,7.31,2.23,6.22,SR
4,Jesse Mendez,Ohio State,1,2,[141],19,19,0,100.0,17,89.5,9,3,5,2,57.37,16.64,12.36,4.25,SR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,Alaa Aly,Edinboro,208,47,[184],1,0,1,0.0,0,0.0,0,0,0,0,40.00,5.00,-15.00,0.00,RSFR
1502,Cole Tolley,West Virginia,114,17,[197],1,0,1,0.0,0,0.0,0,0,0,0,63.00,6.00,-15.00,0.00,SO
1503,Winston McBride,Wyoming,182,21,[285],1,1,0,100.0,0,0.0,0,0,0,1,240.00,1.00,1.00,0.00,SO
1504,Aiden Robertson,Air Force,222,52,[174],1,0,1,0.0,0,0.0,0,0,0,0,76.00,0.00,-5.00,0.00,FR


In [16]:
def filter_weights(wrestlers: list, match_details=False):
  if match_details:
    print("Matches Detailed")
    display(matches_reference.query('home_name in @wrestlers or away_name in @wrestlers')); print()

  print("Overall Stats")
  display(inference_pipeline_db.query('wrestler_name in @wrestlers')); print()

wrestlers_125 = ["Eddie Ventresca", "Kysen Terukina", "Tyler Chappell", "Vincent Robinson", "Keyveon Roller", "Spencer Von Savoye", "Nico Provo"]
filter_weights(wrestlers_125)

Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
167,Vincent Robinson,NC State,7,12,[125],15,12,3,80.0,6,40.0,3,3,0,6,56.80,8.33,5.53,6.24,SO
227,Tyler Chappell,Pittsburgh,43,18,[125],14,5,9,35.7,3,21.4,1,2,0,2,56.50,4.86,-1.14,7.68,SO
298,Eddie Ventresca,Virginia Tech,4,9,[125],13,11,2,84.6,4,30.8,3,1,0,7,46.46,8.46,5.69,6.25,JR
437,Keyveon Roller,Virginia,40,34,[125],11,7,4,63.6,4,36.4,0,3,1,3,97.91,7.20,2.70,6.90,JR
485,Spencer Von Savoye,Duke,178,72,[125],11,2,9,18.2,1,9.1,0,1,0,1,103.55,3.45,-9.18,8.21,SO
521,Kysen Terukina,North Carolina,29,29,[125],10,7,3,70.0,1,10.0,0,1,0,6,38.10,4.80,1.70,4.22,SR
636,Nico Provo,Stanford,9,15,[125],9,6,3,66.7,0,0.0,0,0,0,6,27.22,5.11,1.11,4.04,RSJR


In [17]:
# -- Create for all weights
wrestlers_133 = ["Aaron Seidel", "Evan Tallmadge", "Ethan Oakley", "Zach Redding", "Marlon Yarbrough", "Riley Rowan", "Tyler Knox"]
print("133 ACC WRESTLERS")
filter_weights(wrestlers_133); print(); print()

wrestlers_141 = ["Jack Consiglio", "Tom Crook", "Briar Priest", "Ryan Jack", "Gable Porter", "Raymond Adams", "Luke Simcox"]
print("141 ACC WRESTLERS")
filter_weights(wrestlers_141); print(); print()

wrestlers_149 = ["Koy Buesgens", "Kade Brown", "Wynton Denkins", "Collin Gaj", "Dylan Ross", "Nate Askew", "Aden Valencia"]
print("149 ACC WRESTLERS")
filter_weights(wrestlers_149); print(); print()

wrestlers_157 = ["Daniel Cardenas", "Laird Root", "Colton Washleski", "Dylan Evans", "Luca Felix", "Mikey Boulanger", "Ethen Miller"]
print("157 ACC WRESTLERS")
filter_weights(wrestlers_157); print(); print()

wrestlers_165 = ["Will Denny", "Bryce Hepner", "Jared Keslar", "Mac Church", "Michael Murphy", "Aurelius Dunbar", "EJ Parco"]
print("165 ACC WRESTLERS")
filter_weights(wrestlers_165); print(); print()

wrestlers_174 = ["Luca Augustine", "Sergio Desiante", "Nick Hamilton", "Aiden Wallace", "Collin Carrigan", "Collin Guffey", "Matthew Singleton"]
print("174 ACC WRESTLERS")
filter_weights(wrestlers_174); print(); print()

wrestlers_184 = ["Chase Kranitz", "Jake Dailey", "Donald Cates", "Jaden Bullock", "Griffin Gammell", "David Hussey", "Abe Wojcikiewicz"]
print("184 ACC WRESTLERS")
filter_weights(wrestlers_184); print(); print()

wrestlers_197 = ["Angelo Posada", "Patrick Brophy", "Robert Platt", "Sonny Sasso", "Steven Burrell", "Owen McGrory", "Mac Stout"]
print("197 ACC WRESTLERS")
filter_weights(wrestlers_197); print(); print()


wrestlers_285 = ["Isaac Trumble", "Connor Barket", "Brenan Morgan", "Jim Mullen", "Jacob Levy", "Luke Duthie", "Dayton Pitzer"]
print("285 ACC WRESTLERS")
filter_weights(wrestlers_285); print(); print()

all_wrestler_list = [wrestlers_125, wrestlers_133, wrestlers_141, wrestlers_149, wrestlers_157, wrestlers_165, wrestlers_174, wrestlers_184, wrestlers_197, wrestlers_285]
weights = [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]

133 ACC WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
75,Riley Rowan,Duke,221,72,"[125, 133]",16,2,14,12.5,0,0.0,0,0,0,2,90.31,2.08,-10.83,6.19,RSFR
174,Zach Redding,NC State,44,12,[133],15,9,6,60.0,5,33.3,3,2,0,4,78.93,7.60,3.07,8.91,SR
208,Evan Tallmadge,Pittsburgh,50,18,[133],14,8,6,57.1,3,21.4,3,0,0,5,78.86,6.93,3.14,7.81,SO
337,Tyler Knox,Stanford,21,15,[133],12,10,2,83.3,3,25.0,0,1,2,7,71.17,5.89,2.33,6.10,SO
508,Ethan Oakley,North Carolina,42,29,[133],10,5,5,50.0,2,20.0,0,1,1,3,68.20,4.56,0.22,5.43,SR
867,Aaron Seidel,Virginia Tech,8,9,[133],5,4,1,80.0,4,80.0,3,1,0,0,28.40,13.60,11.20,6.94,FR
976,Marlon Yarbrough,Virginia,46,34,[133],4,1,3,25.0,0,0.0,0,0,0,1,24.25,2.75,-6.50,6.61,SR





141 ACC WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
16,Ryan Jack,NC State,17,12,[141],18,12,6,66.7,8,44.4,2,3,3,4,72.78,8.27,3.20,7.79,RSSR
34,Raymond Adams,Duke,161,72,[141],18,3,15,16.7,1,5.6,1,0,0,2,91.78,3.24,-5.41,7.92,JR
370,Gable Porter,Virginia,23,34,[141],12,9,3,75.0,6,50.0,2,3,1,3,86.17,8.73,4.36,8.66,SO
380,Tom Crook,Virginia Tech,37,9,[141],12,5,7,41.7,3,25.0,1,2,0,2,46.67,6.82,1.36,7.17,RSJR
416,Briar Priest,Pittsburgh,51,18,[141],12,7,5,58.3,1,8.3,1,0,0,6,64.83,4.75,1.33,5.55,JR
440,Jack Consiglio,Stanford,12,15,[141],11,7,4,63.6,3,27.3,0,2,1,4,43.27,6.00,0.44,8.83,RSFR
605,Luke Simcox,North Carolina,29,29,[141],9,5,4,55.6,2,22.2,1,0,1,3,55.44,4.57,1.00,8.16,RSFR





149 ACC WRESTLERS
Overall Stats


...

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
135,David Hussey,Duke,200,72,[184],15,1,14,6.7,0,0.0,0,0,0,1,75.53,3.13,-10.00,6.02,SO
171,Donald Cates,NC State,98,12,[184],15,5,10,33.3,2,13.3,0,0,2,3,68.93,4.46,-3.62,6.28,SR
180,Chase Kranitz,Pittsburgh,35,18,[184],15,10,5,66.7,3,20.0,1,2,0,7,64.13,7.07,2.64,6.87,JR
243,Jaden Bullock,Virginia Tech,30,9,[184],14,7,7,50.0,2,14.3,1,1,0,5,57.07,6.07,-0.64,8.23,SR
329,Abe Wojcikiewicz,Stanford,41,15,[184],13,7,6,53.8,4,30.8,1,2,1,3,81.38,6.83,2.00,6.61,SO
487,Jake Dailey,North Carolina,27,29,[184],11,10,1,90.9,4,36.4,2,2,0,6,70.36,10.09,5.91,6.61,RSFR
784,Griffin Gammell,Virginia,89,34,[184],7,2,5,28.6,0,0.0,0,0,0,2,64.57,5.71,-5.14,6.64,JR





197 ACC WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
74,Owen McGrory,Duke,169,72,[197],16,2,14,12.5,1,6.2,1,0,0,1,75.00,4.81,-7.00,7.88,SO
181,Mac Stout,Pittsburgh,6,18,[197],15,12,3,80.0,5,33.3,3,2,0,7,57.67,9.13,5.80,6.10,JR
210,Sonny Sasso,Virginia Tech,8,9,[197],14,12,2,85.7,7,50.0,1,6,0,5,52.50,9.71,5.00,6.71,SO
338,Robert Platt,North Carolina,46,29,[197],12,6,6,50.0,4,33.3,3,1,0,2,60.00,7.50,2.17,9.48,RSFR
361,Steven Burrell,Virginia,76,34,[197],12,7,5,58.3,2,16.7,1,1,0,5,68.67,7.91,0.73,9.07,SO
408,Angelo Posada,Stanford,5,15,[197],12,10,2,83.3,6,50.0,2,1,3,4,52.58,8.00,3.44,9.13,FR
496,Patrick Brophy,NC State,43,12,[197],11,7,4,63.6,2,18.2,1,1,0,5,94.27,9.50,3.80,6.39,JR





285 ACC WRESTLERS
Overall Stats


,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
122,Dayton Pitzer,Pittsburgh,10,18,[285],15,11,4,73.3,7,46.7,1,3,3,4,58.07,6.33,4.17,6.83,JR
145,Connor Barket,Duke,29,72,[285],15,12,3,80.0,9,60.0,2,6,1,3,78.53,9.86,6.21,7.47,JR
231,Brenan Morgan,Virginia,58,34,[285],14,8,6,57.1,5,35.7,2,0,3,3,77.21,8.11,2.44,8.59,RSFR
261,Isaac Trumble,NC State,2,12,[285],13,13,0,100.0,10,76.9,3,3,4,3,78.62,11.89,9.22,4.68,SR
276,Jim Mullen,Virginia Tech,19,9,[285],13,8,5,61.5,4,30.8,2,1,1,4,56.31,7.25,3.00,7.01,SO
752,Luke Duthie,Stanford,112,15,[285],7,1,6,14.3,1,14.3,1,0,0,0,45.00,7.40,-3.80,12.36,JR
826,Jacob Levy,North Carolina,44,29,[285],6,3,3,50.0,1,16.7,0,0,1,2,74.17,4.80,-5.60,9.94,FR


In [19]:
# -- 1 -- BASIC LOGREG (win_rate difference)
with open("logreg_model_OFFICIAL.pkl", "rb") as f:
    logreg_model = pickle.load(f)

# -- 2 -- DECISION TREE
with open("dt_model_tuned_OFFICIAL.pkl", "rb") as f:
    dt_model = pickle.load(f)

# -- 3 -- XGBoost w/o ranks
with open("xgboost_model_tuned_noranks_OFFICIAL.pkl", "rb") as f:
    xgb_model1 = pickle.load(f)

# -- 4 -- XGbosst w rank
with open("xgb_with_ranks_tuned_OFFICIAL.pkl", "rb") as f:
    xgb_model2 = pickle.load(f)



# -- Features Dt and XGBoost(1)
with open("features_dt_xgb1.pkl", "rb") as f:
    features_dt_xgb1 = pickle.load(f)


# -- Features XGBoost(2)
with open("xgb_w_ranks_features.pkl", "rb") as f:
    features_dt_xgb2 = pickle.load(f)

In [20]:
# -- Demonstrate pred -- This is Aaron Sidel vs Tyler Knox Pred
logreg_model.predict_proba(np.array([-0.03]).reshape(-1, 1))

array([[0.51377978, 0.48622022]])

In [ ]:
matches_reference.query('home_name == "Collin Gaj" or away_name == "Collin Gaj"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
515,210,149,11/15/2025,238.0,Koy Buesgens,22.0,307.0,Collin Gaj,4.0,True,WDEC,WDEC 4 - 1,SO,NC State,FR,Virginia Tech
566,214,157,11/15/2025,307.0,Collin Gaj,4.0,972.0,Jared Hill,72.0,False,LSV,LSV-1 9 - 6,FR,Virginia Tech,SR,Wyoming
794,193,157,11/16/2025,307.0,Collin Gaj,4.0,143.0,Kannon Webster,8.0,False,LTF,LTF5 19 - 4 7:00,FR,Virginia Tech,SO,Illinois
837,191,157,11/16/2025,153.0,Charlie Millard,15.0,307.0,Collin Gaj,4.0,True,WDEC,WDEC 9 - 4,RSFR,Minnesota,FR,Virginia Tech
2580,34,149,1/9/2026,297.0,Kade Brown,35.0,307.0,Collin Gaj,4.0,False,LDEC,LDEC 5 - 4,RSFR,Pittsburgh,FR,Virginia Tech
3110,328,149,1/16/2026,307.0,Collin Gaj,4.0,406.0,Kaden Keiser,50.0,True,WMD,WMD 10 - 2,FR,Virginia Tech,JR,Appalachian State
3607,375,149,1/23/2026,307.0,Collin Gaj,4.0,427.0,Aden Valencia,21.0,True,WDEC,WDEC 2 - 1,FR,Virginia Tech,RSFR,Stanford
4071,415,149,1/30/2026,238.0,Koy Buesgens,12.0,307.0,Collin Gaj,4.0,False,LDEC,LDEC 6 - 0,SO,NC State,FR,Virginia Tech
4553,520,149,2/6/2026,248.0,Wynton Denkins,34.0,307.0,Collin Gaj,5.0,True,WDEC,WDEC 3 - 2,JR,Virginia,FR,Virginia Tech


In [21]:
inference_pipeline_db.query('wrestler_name == "Collin Gaj"')

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
596,Collin Gaj,Virginia Tech,6,9,"[149, 157]",9,4,5,44.4,1,11.1,0,1,0,3,29.89,4.44,-1.22,6.69,FR


In [22]:
inference_pipeline_db.query('wrestler_name == "Koy Buesgens"')

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
67,Koy Buesgens,NC State,5,12,[149],17,15,2,88.2,7,41.2,1,5,1,8,60.76,9.19,5.44,5.63,SO


In [ ]:
features_dt_xgb2

Index(['wrestled_before', 'home_point_diff_rematches', 'home_pinned_away',
       'matches_wrestled_diff', 'win_rate_diff', 'bonus_rate_diff',
       'pin_count_diff', 'avg_point_diff_diff', 'avg_points_scored_diff',
       'std_point_diff_diff', 'team_rank_diff', 'weight_class_133',
       'weight_class_141', 'weight_class_149', 'weight_class_157',
       'weight_class_165', 'weight_class_174', 'weight_class_184',
       'weight_class_197', 'weight_class_285', 'home_class_JR',
       'home_class_RSFR', 'home_class_RSJR', 'home_class_RSSO',
       'home_class_RSSR', 'home_class_SO', 'home_class_SR', 'away_class_JR',
       'away_class_RSFR', 'away_class_RSJR', 'away_class_RSSO',
       'away_class_RSSR', 'away_class_SO', 'away_class_SR'],
      dtype='object')

# Example on one pair of wrestlers

In [23]:
features_xgb2 = features_dt_xgb2.tolist()

# ============================================
# FUNCTION TO GET MATCHUP HISTORY
# ============================================

def get_wrestler_match_history(wrestler1_name, wrestler2_name, matches_df):
    """
    Get complete match history between two wrestlers from matches_reference.
    Returns DataFrame of all matches and calculated historical features.
    """

    # Find all matches between these two wrestlers
    mask = (
        ((matches_df['home_name'] == wrestler1_name) & (matches_df['away_name'] == wrestler2_name)) |
        ((matches_df['home_name'] == wrestler2_name) & (matches_df['away_name'] == wrestler1_name))
    )

    history = matches_df[mask].copy().sort_values('event_date')

    if len(history) == 0:
        return pd.DataFrame(), {
            'wrestled_before': 0,
            'home_point_diff_rematches': 0,
            'home_pinned_away': 0,
            'total_matches': 0,
            'w1_wins': 0,
            'w2_wins': 0,
            'avg_point_diff_w1': 0,
            'last_match_date': None,
            'last_winner': None
        }

    # Calculate head-to-head stats
    w1_name = wrestler1_name
    w2_name = wrestler2_name

    w1_wins = 0
    w2_wins = 0
    total_point_diff_w1 = 0
    point_diff_count = 0
    pins_by_w1 = 0
    pins_by_w2 = 0

    # Add columns for analysis
    history['winner'] = None
    history['point_diff'] = None
    history['was_pin'] = False

    for idx, row in history.iterrows():
        # Determine winner
        if row['home_name'] == w1_name:
            winner = w1_name if row['home_win'] else w2_name
        else:
            winner = w2_name if row['home_win'] else w1_name

        history.loc[idx, 'winner'] = winner

        # Calculate point differential (from w1's perspective)
        if 'FALL' not in str(row['win_type']):
            try:
                parts = str(row['Result']).split()
                scores = [int(x) for x in parts if x.isdigit()]
                if len(scores) >= 2:
                    score1, score2 = scores[0], scores[1]

                    # Determine scores from w1's perspective
                    if row['home_name'] == w1_name:
                        if row['home_win']:
                            point_diff = score1 - score2
                        else:
                            point_diff = score2 - score1
                    else:
                        if not row['home_win']:
                            point_diff = score1 - score2
                        else:
                            point_diff = score2 - score1

                    history.loc[idx, 'point_diff'] = point_diff
                    total_point_diff_w1 += point_diff
                    point_diff_count += 1
            except:
                pass

        # Check for pins
        if 'FALL' in str(row['win_type']):
            history.loc[idx, 'was_pin'] = True
            if winner == w1_name:
                pins_by_w1 += 1
            else:
                pins_by_w2 += 1

        # Count wins
        if winner == w1_name:
            w1_wins += 1
        else:
            w2_wins += 1

    # Calculate averages
    avg_point_diff_w1 = total_point_diff_w1 / point_diff_count if point_diff_count > 0 else 0

    # Historical features for next matchup
    historical_features = {
        'wrestled_before': 1 if len(history) > 0 else 0,
        'home_point_diff_rematches': round(avg_point_diff_w1, 3),
        'home_pinned_away': 1 if pins_by_w1 > 0 else 0,
        'total_matches': len(history),
        'w1_wins': w1_wins,
        'w2_wins': w2_wins,
        'avg_point_diff_w1': round(avg_point_diff_w1, 3),
        'pins_by_w1': pins_by_w1,
        'pins_by_w2': pins_by_w2,
        'last_match_date': history.iloc[-1]['event_date'] if len(history) > 0 else None,
        'last_winner': history.iloc[-1]['winner'] if len(history) > 0 else None
    }

    return history, historical_features

# ============================================
# FUNCTION TO PREPARE FEATURES FOR A MATCHUP
# ============================================

def prepare_matchup_features(w1_name, w2_name, historical_features, inference_db, weight_class=125):
    """
    Prepare all features needed for models given two wrestlers.
    """

    # Get wrestler data from inference pipeline
    w1_data = inference_db[inference_db['wrestler_name'] == w1_name].iloc[0]
    w2_data = inference_db[inference_db['wrestler_name'] == w2_name].iloc[0]

    # Determine home/away: Virginia Tech always home, otherwise by rank
    if w1_data['team_name'] == 'Virginia Tech':
        home, away = w1_data, w2_data
        home_name, away_name = w1_name, w2_name
    elif w2_data['team_name'] == 'Virginia Tech':
        home, away = w2_data, w1_data
        home_name, away_name = w2_name, w1_name
    elif w1_data['rank'] < w2_data['rank']:
        home, away = w1_data, w2_data
        home_name, away_name = w1_name, w2_name
    else:
        home, away = w2_data, w1_data
        home_name, away_name = w2_name, w1_name

    print(f"\n📌 Home wrestler: {home_name} (Rank: {home['rank']}, Team: {home['team_name']})")
    print(f"📌 Away wrestler: {away_name} (Rank: {away['rank']}, Team: {away['team_name']})")

    # Calculate differential features
    features = {}

    # Basic differentials
    features['matches_wrestled_diff'] = home['total_matches'] - away['total_matches']
    features['win_rate_diff'] = round(home['win_rate']/100 - away['win_rate']/100, 4)
    features['bonus_rate_diff'] = round(home['bonus_rate']/100 - away['bonus_rate']/100, 4)
    features['pin_count_diff'] = home['fall_wins'] - away['fall_wins']
    features['avg_point_diff_diff'] = round(home['avg_point_diff'] - away['avg_point_diff'], 4)
    features['avg_points_scored_diff'] = round(home['avg_points_scored'] - away['avg_points_scored'], 4)
    features['std_point_diff_diff'] = round(home['std_point_diff'] - away['std_point_diff'], 4)
    features['team_rank_diff'] = home['team_rank'] - away['team_rank']

    # Historical features
    features['wrestled_before'] = historical_features['wrestled_before']
    features['home_point_diff_rematches'] = historical_features['home_point_diff_rematches']
    features['home_pinned_away'] = historical_features['home_pinned_away']

    # Weight class dummies
    weight_classes = [133, 141, 149, 157, 165, 174, 184, 197, 285]
    for wc in weight_classes:
        features[f'weight_class_{wc}'] = 1 if weight_class == wc else 0

    # Class dummies
    all_classes = ['JR', 'RSFR', 'RSJR', 'RSSO', 'RSSR', 'SO', 'SR']

    for cls in all_classes:
        features[f'home_class_{cls}'] = 1 if home['Class'] == cls else 0

    for cls in all_classes:
        features[f'away_class_{cls}'] = 1 if away['Class'] == cls else 0

    return features, home_name, away_name

# ============================================
# FUNCTION TO MAKE PREDICTIONS
# ============================================

def predict_matchup(w1_name, w2_name, features, home_name, away_name):
    """
    Make predictions using all 4 models.
    """

    # Create dataframes with correct feature order
    df_dt = pd.DataFrame([{f: features.get(f, 0) for f in features_dt_xgb1}])
    df_xgb2 = pd.DataFrame([{f: features.get(f, 0) for f in features_xgb2}])

    print("\n" + "="*60)
    print("FEATURE ARRAYS BEING PASSED TO MODELS")
    print("="*60)

    # LOGREG features
    logreg_input = features['win_rate_diff']
    print(f"\n📊 LOGREG input: [{logreg_input}]")

    # DT/XGB1 features (show all)
    print(f"\n📊 DT/XGB1 features ({len(features_dt_xgb1)} total):")
    dt_features_dict = {f: round(df_dt[f].iloc[0], 4) for f in features_dt_xgb1}
    for i, (feat, val) in enumerate(dt_features_dict.items()):
        print(f"   {i+1:2d}. {feat:35} = {val:8.4f}")

    # XGB2 features (show all)
    print(f"\n📊 XGB2 features ({len(features_xgb2)} total):")
    xgb2_features_dict = {f: round(df_xgb2[f].iloc[0], 4) for f in features_xgb2}
    for i, (feat, val) in enumerate(xgb2_features_dict.items()):
        print(f"   {i+1:2d}. {feat:35} = {val:8.4f}")

    # Make predictions
    results = {}

    # LOGREG
    logreg_input_array = [[logreg_input]]
    results['logreg_pred'] = logreg_model.predict(logreg_input_array)[0]
    results['logreg_prob'] = round(logreg_model.predict_proba(logreg_input_array)[0][1], 4)
    results['logreg_winner'] = home_name if results['logreg_pred'] == 1 else away_name

    # DT
    dt_input = df_dt[features_dt_xgb1]
    results['dt_pred'] = dt_model.predict(dt_input)[0]
    results['dt_prob'] = round(dt_model.predict_proba(dt_input)[0][1], 4)
    results['dt_winner'] = home_name if results['dt_pred'] == 1 else away_name

    # XGB1
    xgb1_input = df_dt[features_dt_xgb1]
    results['xgb1_pred'] = xgb_model1.predict(xgb1_input)[0]
    results['xgb1_prob'] = round(xgb_model1.predict_proba(xgb1_input)[0][1], 4)
    results['xgb1_winner'] = home_name if results['xgb1_pred'] == 1 else away_name

    # XGB2
    xgb2_input = df_xgb2[features_xgb2]
    results['xgb2_pred'] = xgb_model2.predict(xgb2_input)[0]
    results['xgb2_prob'] = round(xgb_model2.predict_proba(xgb2_input)[0][1], 4)
    results['xgb2_winner'] = home_name if results['xgb2_pred'] == 1 else away_name

    return results

# ============================================
# ANALYZE NICO PROVO VS KYSEN TERUKINA
# ============================================

print("\n" + "="*80)
print("ANALYZING: NICO PROVO vs KYSEN TERUKINA")
print("="*80)

# Get their match history
history, hist_features = get_wrestler_match_history(
    "Nico Provo", "Kysen Terukina", matches_reference
)

print(f"\n📜 Match History:")
if len(history) > 0:
    for idx, row in history.iterrows():
        print(f"   {row['event_date']}: {row['home_name']} vs {row['away_name']} - {row['win_type']} {row['Result']}")
        print(f"      Winner: {row['winner']}, Point Diff: {row['point_diff']}, Pin: {row['was_pin']}")

    print(f"\n📊 Historical Summary:")
    print(f"   Total matches: {hist_features['total_matches']}")
    print(f"   Nico Provo wins: {hist_features['w1_wins']}")
    print(f"   Kysen Terukina wins: {hist_features['w2_wins']}")
    print(f"   Avg point diff (Nico's perspective): {hist_features['avg_point_diff_w1']}")
    print(f"   Pins by Nico: {hist_features['pins_by_w1']}")
    print(f"   Pins by Kysen: {hist_features['pins_by_w2']}")
    print(f"   Last match: {hist_features['last_match_date']} (Winner: {hist_features['last_winner']})")
else:
    print("   No previous matches found")

print(f"\n🔍 Historical Features for Next Match:")
print(f"   wrestled_before: {hist_features['wrestled_before']}")
print(f"   home_point_diff_rematches: {hist_features['home_point_diff_rematches']}")
print(f"   home_pinned_away: {hist_features['home_pinned_away']}")

# Prepare features for prediction
features, home_name, away_name = prepare_matchup_features(
    "Nico Provo", "Kysen Terukina", hist_features, inference_pipeline_db, weight_class=125
)

# Make predictions
results = predict_matchup("Nico Provo", "Kysen Terukina", features, home_name, away_name)

# Display results
print("\n" + "="*60)
print("PREDICTION RESULTS")
print("="*60)

print(f"\n🏆 Match: {home_name} (HOME) vs {away_name} (AWAY)")
print(f"\n{'Model':<20} {'Predicted Winner':<25} {'Probability':<15}")
print("-" * 60)
print(f"{'Logistic Regression':<20} {results['logreg_winner']:<25} {results['logreg_prob']:<15}")
print(f"{'Decision Tree':<20} {results['dt_winner']:<25} {results['dt_prob']:<15}")
print(f"{'XGBoost (no ranks)':<20} {results['xgb1_winner']:<25} {results['xgb1_prob']:<15}")
print(f"{'XGBoost (with ranks)':<20} {results['xgb2_winner']:<25} {results['xgb2_prob']:<15}")

# Check if all models agree
all_winners = [results['logreg_winner'], results['dt_winner'],
               results['xgb1_winner'], results['xgb2_winner']]
if all(w == all_winners[0] for w in all_winners):
    print(f"\n✅ All models agree: {all_winners[0]} wins!")
else:
    print(f"\n⚠️ Models disagree on winner")




# ============================================
# ADD NEW TEST CASE: COLLIN GAJ VS KOY BUESGENS
# ============================================

print("\n" + "="*80)
print("🔍 TEST CASE: COLLIN GAJ vs KOY BUESGENS")
print("="*80)

# Get their match history
history_gaj, hist_features_gaj = get_wrestler_match_history(
    "Collin Gaj", "Koy Buesgens", matches_reference
)

print(f"\n📜 Match History:")
if len(history_gaj) > 0:
    for idx, row in history_gaj.iterrows():
        print(f"   {row['event_date']}: {row['home_name']} vs {row['away_name']} - {row['win_type']} {row['Result']}")
        print(f"      Winner: {row['winner']}, Point Diff: {row['point_diff']}, Pin: {row['was_pin']}")

    print(f"\n📊 Historical Summary:")
    print(f"   Total matches: {hist_features_gaj['total_matches']}")
    print(f"   Collin Gaj wins: {hist_features_gaj['w1_wins']}")
    print(f"   Koy Buesgens wins: {hist_features_gaj['w2_wins']}")
    print(f"   Avg point diff (Collin's perspective): {hist_features_gaj['avg_point_diff_w1']}")
    print(f"   Pins by Collin: {hist_features_gaj['pins_by_w1']}")
    print(f"   Pins by Koy: {hist_features_gaj['pins_by_w2']}")
    print(f"   Last match: {hist_features_gaj['last_match_date']} (Winner: {hist_features_gaj['last_winner']})")

    # Verify historical features match expected values
    print(f"\n🔍 Historical Features Verification:")
    print(f"   wrestled_before: {hist_features_gaj['wrestled_before']} (expected: 1)")
    print(f"   home_point_diff_rematches: {hist_features_gaj['home_point_diff_rematches']} (expected: 1.5)")
    print(f"   home_pinned_away: {hist_features_gaj['home_pinned_away']} (expected: 0)")
else:
    print("   No previous matches found")

# Prepare features for prediction
print(f"\n📊 Wrestler Data from Inference Pipeline:")
collin_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Collin Gaj'].iloc[0]
koy_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Koy Buesgens'].iloc[0]

print(f"   Collin Gaj - Rank: {collin_data['rank']}, Team: {collin_data['team_name']}, Win Rate: {collin_data['win_rate']}%")
print(f"   Koy Buesgens - Rank: {koy_data['rank']}, Team: {koy_data['team_name']}, Win Rate: {koy_data['win_rate']}%")

# Calculate win_rate_diff for LOGREG
win_rate_diff = collin_data['win_rate']/100 - koy_data['win_rate']/100
print(f"\n📊 LOGREG Input (win_rate_diff): {win_rate_diff:.4f} (Collin {collin_data['win_rate']}% - Koy {koy_data['win_rate']}%)")

# Prepare features for prediction
features_gaj, home_name_gaj, away_name_gaj = prepare_matchup_features(
    "Collin Gaj", "Koy Buesgens", hist_features_gaj, inference_pipeline_db, weight_class=149
)

# Make predictions
results_gaj = predict_matchup("Collin Gaj", "Koy Buesgens", features_gaj, home_name_gaj, away_name_gaj)

# Display results
print("\n" + "="*60)
print("PREDICTION RESULTS - COLLIN GAJ vs KOY BUESGENS")
print("="*60)

print(f"\n🏆 Match: {home_name_gaj} (HOME) vs {away_name_gaj} (AWAY)")
print(f"\n{'Model':<20} {'Predicted Winner':<25} {'Probability':<15}")
print("-" * 60)
print(f"{'Logistic Regression':<20} {results_gaj['logreg_winner']:<25} {results_gaj['logreg_prob']:<15}")
print(f"{'Decision Tree':<20} {results_gaj['dt_winner']:<25} {results_gaj['dt_prob']:<15}")
print(f"{'XGBoost (no ranks)':<20} {results_gaj['xgb1_winner']:<25} {results_gaj['xgb1_prob']:<15}")
print(f"{'XGBoost (with ranks)':<20} {results_gaj['xgb2_winner']:<25} {results_gaj['xgb2_prob']:<15}")

# Check if all models agree
all_winners_gaj = [results_gaj['logreg_winner'], results_gaj['dt_winner'],
                   results_gaj['xgb1_winner'], results_gaj['xgb2_winner']]
if all(w == all_winners_gaj[0] for w in all_winners_gaj):
    print(f"\n✅ All models agree: {all_winners_gaj[0]} wins!")
else:
    print(f"\n⚠️ Models disagree on winner")

# ============================================
# BONUS: DRAKE AYALA VS BEN DAVINO
# ============================================



# ============================================
# BONUS: DRAKE AYALA VS BEN DAVINO
# ============================================

print("\n" + "="*80)
print("BONUS: DRAKE AYALA vs BEN DAVINO")
print("="*80)

history2, hist_features2 = get_wrestler_match_history(
    "Drake Ayala", "Ben Davino", matches_reference
)

print(f"\n📜 Match History:")
if len(history2) > 0:
    for idx, row in history2.iterrows():
        print(f"   {row['event_date']}: {row['home_name']} vs {row['away_name']} - {row['win_type']} {row['Result']}")
        print(f"      Winner: {row['winner']}, Point Diff: {row['point_diff']}, Pin: {row['was_pin']}")

    print(f"\n📊 Historical Summary:")
    print(f"   Total matches: {hist_features2['total_matches']}")
    print(f"   Drake Ayala wins: {hist_features2['w1_wins'] if hist_features2['w1_wins'] > 0 else 0}")
    print(f"   Ben Davino wins: {hist_features2['w2_wins'] if hist_features2['w2_wins'] > 0 else 0}")
    print(f"   Avg point diff (Drake's perspective): {hist_features2['avg_point_diff_w1']}")
    print(f"   Pins by Drake: {hist_features2['pins_by_w1']}")
    print(f"   Pins by Ben: {hist_features2['pins_by_w2']}")

    print(f"\n🔍 Historical Features for Next Match:")
    print(f"   wrestled_before: {hist_features2['wrestled_before']}")
    print(f"   home_point_diff_rematches: {hist_features2['home_point_diff_rematches']}")
    print(f"   home_pinned_away: {hist_features2['home_pinned_away']}")
else:
    print("   No previous matches found")


ANALYZING: NICO PROVO vs KYSEN TERUKINA

📜 Match History:
   2/20/2026: Kysen Terukina vs Nico Provo - LDEC LDEC 4 - 1
      Winner: Nico Provo, Point Diff: 3, Pin: False

📊 Historical Summary:
   Total matches: 1
   Nico Provo wins: 1
   Kysen Terukina wins: 0
   Avg point diff (Nico's perspective): 3.0
   Pins by Nico: 0
   Pins by Kysen: 0
   Last match: 2/20/2026 (Winner: Nico Provo)

🔍 Historical Features for Next Match:
   wrestled_before: 1
   home_point_diff_rematches: 3.0
   home_pinned_away: 0

📌 Home wrestler: Nico Provo (Rank: 9, Team: Stanford)
📌 Away wrestler: Kysen Terukina (Rank: 29, Team: North Carolina)

FEATURE ARRAYS BEING PASSED TO MODELS

📊 LOGREG input: [-0.033]

📊 DT/XGB1 features (33 total):
    1. wrestled_before                     =   1.0000
    2. home_point_diff_rematches           =   3.0000
    3. home_pinned_away                    =   0.0000
    4. weight_class_133                    =   0.0000
    5. weight_class_141                    =   0.0000
   

In [24]:
test = ["Nico Provo", "Kysen Terukina"]
wrestlers.query('name in @test')

,wrestler_id,name,Class,Team
423,426,Nico Provo,RSJR,Stanford
640,643,Kysen Terukina,SR,North Carolina


In [25]:
teams.query('name == "North Carolina"')

,team_id,name
54,55,North Carolina


In [26]:
duals.query('team1_id == 55 or team2_id == 55')

,dual_id,team1_id,team1_rank,team1_score,team2_id,team2_rank,team2_score,event_date
66,67,55,23,19,1,14,13,12/21/25
98,99,19,3,27,55,23,9,12/19/25
129,130,55,23,27,30,17,7,12/12/25
178,179,55,23,47,65,71,0,11/21/25
226,227,55,23,20,68,21,18,11/15/25
251,252,55,23,40,49,61,0,11/09/25
337,338,55,25,25,22,41,10,01/16/26
376,377,21,11,24,55,23,10,01/23/26
420,421,27,17,22,55,23,9,01/30/26
470,471,28,7,29,55,27,9,02/13/26


In [27]:
inference_pipeline_db.query('wrestler_name == "Tyler Knox"')

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
337,Tyler Knox,Stanford,21,15,[133],12,10,2,83.3,3,25.0,0,1,2,7,71.17,5.89,2.33,6.1,SO


In [28]:
matches_reference.query('home_name == "Kysen Terukina" or away_name == "Kysen Terukina"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
311,252,125,11/9/2025,643.0,Kysen Terukina,26.0,545.0,Aydan Thomas,77.0,True,WDEC,WDEC 6 - 2,SR,North Carolina,FR,Northern Colorado
733,227,125,11/15/2025,643.0,Kysen Terukina,26.0,836.0,Andrew Binni,37.0,True,WMD,WMD 11 - 1,SR,North Carolina,FR,Navy
1026,179,125,11/21/2025,643.0,Kysen Terukina,26.0,904.0,Brady Joling,115.0,True,WDEC,WDEC 3 - 0,SR,North Carolina,SO,Davidson
1573,130,125,12/12/2025,643.0,Kysen Terukina,26.0,324.0,Louie Gill,29.0,True,WDEC,WDEC 6 - 3,SR,North Carolina,RSFR,West Virginia
1824,99,125,12/19/2025,215.0,Alan Koehler,44.0,643.0,Kysen Terukina,26.0,False,LDEC,LDEC 4 - 1,SO,Nebraska,SR,North Carolina
2052,67,125,12/21/2025,31.0,Trever Anderson,21.0,643.0,Kysen Terukina,26.0,False,LDEC,LDEC 5 - 3,JR,Northern Iowa,SR,North Carolina
3544,377,125,1/23/2026,628.0,Vincent Robinson,7.0,643.0,Kysen Terukina,29.0,True,WDEC,WDEC 4 - 2,SO,NC State,SR,North Carolina
4141,421,125,1/30/2026,643.0,Kysen Terukina,29.0,294.0,Tyler Chappell,39.0,True,WDEC,WDEC 8 - 6,SR,North Carolina,SO,Pittsburgh
4966,471,125,2/13/2026,304.0,Eddie Ventresca,3.0,643.0,Kysen Terukina,31.0,True,WDEC,WDEC 7 - 2,JR,Virginia Tech,SR,North Carolina
5530,574,125,2/20/2026,643.0,Kysen Terukina,29.0,426.0,Nico Provo,9.0,False,LDEC,LDEC 4 - 1,SR,North Carolina,RSJR,Stanford


In [29]:
# -- test again
get_wrestler_match_history(
    "Koy Buesgens", "Collin Gaj", matches_reference
)


(      dual_id  weight_class  event_date  home_wrestler_id     home_name  \
 4071      415           149   1/30/2026             238.0  Koy Buesgens   
 515       210           149  11/15/2025             238.0  Koy Buesgens   
 
       home_rank  away_wrestler_id   away_name  away_rank  home_win win_type  \
 4071       12.0             307.0  Collin Gaj        4.0     False     LDEC   
 515        22.0             307.0  Collin Gaj        4.0      True     WDEC   
 
           Result home_class home_team_name away_class away_team_name  \
 4071  LDEC 6 - 0         SO       NC State         FR  Virginia Tech   
 515   WDEC 4 - 1         SO       NC State         FR  Virginia Tech   
 
             winner point_diff  was_pin  
 4071    Collin Gaj         -6    False  
 515   Koy Buesgens          3    False  ,
 {'wrestled_before': 1,
  'home_point_diff_rematches': -1.5,
  'home_pinned_away': 0,
  'total_matches': 2,
  'w1_wins': 1,
  'w2_wins': 1,
  'avg_point_diff_w1': -1.5,
  'pins_by_

# Time to make predictions

In [30]:
def get_wrestler_match_history(wrestler1_name, wrestler2_name, matches_df):
    """
    Get complete match history between two wrestlers from matches_reference.
    Returns DataFrame of all matches and calculated historical features.
    """

    # Find all matches between these two wrestlers
    mask = (
        ((matches_df['home_name'] == wrestler1_name) & (matches_df['away_name'] == wrestler2_name)) |
        ((matches_df['home_name'] == wrestler2_name) & (matches_df['away_name'] == wrestler1_name))
    )

    history = matches_df[mask].copy().sort_values('event_date')

    if len(history) == 0:
        return {
            'wrestled_before': 0,
            'home_point_diff_rematches': 0,
            'home_pinned_away': 0,
            'total_matches': 0,
            'w1_wins': 0,
            'w2_wins': 0,
            'avg_point_diff_w1': 0,
            'last_match_date': None,
            'last_winner': None
        }

    # Calculate head-to-head stats
    w1_name = wrestler1_name
    w2_name = wrestler2_name

    w1_wins = 0
    w2_wins = 0
    total_point_diff_w1 = 0
    point_diff_count = 0
    pins_by_w1 = 0
    pins_by_w2 = 0

    # Store match results for return
    match_records = []

    for idx, row in history.iterrows():
        # Determine winner
        if row['home_name'] == w1_name:
            winner = w1_name if row['home_win'] else w2_name
        else:
            winner = w2_name if row['home_win'] else w1_name

        # Calculate point differential (from w1's perspective)
        point_diff = 0
        if 'FALL' not in str(row['win_type']):
            try:
                parts = str(row['Result']).split()
                scores = [int(x) for x in parts if x.isdigit()]
                if len(scores) >= 2:
                    score1, score2 = scores[0], scores[1]

                    # Determine scores from w1's perspective
                    if row['home_name'] == w1_name:
                        if row['home_win']:
                            point_diff = score1 - score2
                        else:
                            point_diff = score2 - score1
                    else:
                        if not row['home_win']:
                            point_diff = score1 - score2
                        else:
                            point_diff = score2 - score1

                    total_point_diff_w1 += point_diff
                    point_diff_count += 1
            except:
                point_diff = 0

        # Check for pins
        was_pin = 'FALL' in str(row['win_type'])
        if was_pin:
            if winner == w1_name:
                pins_by_w1 += 1
            else:
                pins_by_w2 += 1

        # Count wins
        if winner == w1_name:
            w1_wins += 1
        else:
            w2_wins += 1

        # Store match record
        match_records.append({
            'date': row['event_date'],
            'home': row['home_name'],
            'away': row['away_name'],
            'winner': winner,
            'point_diff': point_diff,
            'was_pin': was_pin,
            'result': row['Result'],
            'win_type': row['win_type']
        })

    # Calculate averages
    avg_point_diff_w1 = total_point_diff_w1 / point_diff_count if point_diff_count > 0 else 0

    # Add winner column to history DataFrame for return
    history['winner'] = [m['winner'] for m in match_records]

    return {
        'wrestled_before': 1 if len(history) > 0 else 0,
        'home_point_diff_rematches': round(avg_point_diff_w1, 3),
        'home_pinned_away': 1 if pins_by_w1 > 0 else 0,
        'total_matches': len(history),
        'w1_wins': w1_wins,
        'w2_wins': w2_wins,
        'avg_point_diff_w1': round(avg_point_diff_w1, 3),
        'pins_by_w1': pins_by_w1,
        'pins_by_w2': pins_by_w2,
        'last_match_date': history.iloc[-1]['event_date'] if len(history) > 0 else None,
        'last_winner': history.iloc[-1]['winner'] if len(history) > 0 else None,
        'match_history': history
    }

# ============================================
# FUNCTION TO PREPARE FEATURES FOR A MATCHUP
# ============================================

def prepare_matchup_features(w1_name, w2_name, historical_features, inference_db, weight_class=125):
    """
    Prepare all features needed for models given two wrestlers.
    """

    # Get wrestler data from inference pipeline
    w1_data = inference_db[inference_db['wrestler_name'] == w1_name].iloc[0]
    w2_data = inference_db[inference_db['wrestler_name'] == w2_name].iloc[0]

    # Determine home/away: Virginia Tech always home, otherwise by rank
    if w1_data['team_name'] == 'Virginia Tech':
        home, away = w1_data, w2_data
        home_name, away_name = w1_name, w2_name
    elif w2_data['team_name'] == 'Virginia Tech':
        home, away = w2_data, w1_data
        home_name, away_name = w2_name, w1_name
    elif w1_data['rank'] < w2_data['rank']:
        home, away = w1_data, w2_data
        home_name, away_name = w1_name, w2_name
    else:
        home, away = w2_data, w1_data
        home_name, away_name = w2_name, w1_name

    # Calculate differential features
    features = {}

    # Basic differentials
    features['matches_wrestled_diff'] = home['total_matches'] - away['total_matches']
    features['win_rate_diff'] = round(home['win_rate']/100 - away['win_rate']/100, 4)
    features['bonus_rate_diff'] = round(home['bonus_rate']/100 - away['bonus_rate']/100, 4)
    features['pin_count_diff'] = home['fall_wins'] - away['fall_wins']
    features['avg_point_diff_diff'] = round(home['avg_point_diff'] - away['avg_point_diff'], 4)
    features['avg_points_scored_diff'] = round(home['avg_points_scored'] - away['avg_points_scored'], 4)
    features['std_point_diff_diff'] = round(home['std_point_diff'] - away['std_point_diff'], 4)
    features['team_rank_diff'] = home['team_rank'] - away['team_rank']

    # Historical features
    features['wrestled_before'] = historical_features['wrestled_before']
    features['home_point_diff_rematches'] = historical_features['home_point_diff_rematches']
    features['home_pinned_away'] = historical_features['home_pinned_away']

    # Weight class dummies
    weight_classes = [133, 141, 149, 157, 165, 174, 184, 197, 285]
    for wc in weight_classes:
        features[f'weight_class_{wc}'] = 1 if weight_class == wc else 0

    # Class dummies
    all_classes = ['JR', 'RSFR', 'RSJR', 'RSSO', 'RSSR', 'SO', 'SR']

    for cls in all_classes:
        features[f'home_class_{cls}'] = 1 if home['Class'] == cls else 0

    for cls in all_classes:
        features[f'away_class_{cls}'] = 1 if away['Class'] == cls else 0

    return features, home_name, away_name, home, away

# ============================================
# FIXED FUNCTION TO MAKE PREDICTIONS FOR ONE MATCHUP
# ============================================

def predict_single_matchup(w1_name, w2_name, features, home_name, away_name):
    """
    Make predictions using all 4 models for a single matchup.
    Returns results with winner and winner probability.
    """

    # Create dataframes with correct feature order
    df_dt = pd.DataFrame([{f: features.get(f, 0) for f in features_dt_xgb1}])
    df_xgb2 = pd.DataFrame([{f: features.get(f, 0) for f in features_xgb2}])

    # Make predictions
    results = {
        'wrestler1': w1_name,
        'wrestler2': w2_name,
        'home_wrestler': home_name,
        'away_wrestler': away_name
    }

    # LOGREG
    logreg_input = [[features['win_rate_diff']]]
    logreg_proba = logreg_model.predict_proba(logreg_input)[0]  # [away_prob, home_prob]
    logreg_pred = logreg_model.predict(logreg_input)[0]  # 1 = home wins, 0 = away wins

    results['logreg_pred'] = logreg_pred
    results['logreg_home_prob'] = round(logreg_proba[1], 4)
    results['logreg_away_prob'] = round(logreg_proba[0], 4)
    results['logreg_winner'] = home_name if logreg_pred == 1 else away_name
    results['logreg_winner_prob'] = round(logreg_proba[1] if logreg_pred == 1 else logreg_proba[0], 4)

    # DT
    dt_input = df_dt[features_dt_xgb1]
    dt_proba = dt_model.predict_proba(dt_input)[0]
    dt_pred = dt_model.predict(dt_input)[0]

    results['dt_pred'] = dt_pred
    results['dt_home_prob'] = round(dt_proba[1], 4)
    results['dt_away_prob'] = round(dt_proba[0], 4)
    results['dt_winner'] = home_name if dt_pred == 1 else away_name
    results['dt_winner_prob'] = round(dt_proba[1] if dt_pred == 1 else dt_proba[0], 4)

    # XGB1
    xgb1_input = df_dt[features_dt_xgb1]
    xgb1_proba = xgb_model1.predict_proba(xgb1_input)[0]
    xgb1_pred = xgb_model1.predict(xgb1_input)[0]

    results['xgb1_pred'] = xgb1_pred
    results['xgb1_home_prob'] = round(xgb1_proba[1], 4)
    results['xgb1_away_prob'] = round(xgb1_proba[0], 4)
    results['xgb1_winner'] = home_name if xgb1_pred == 1 else away_name
    results['xgb1_winner_prob'] = round(xgb1_proba[1] if xgb1_pred == 1 else xgb1_proba[0], 4)

    # XGB2
    xgb2_input = df_xgb2[features_xgb2]
    xgb2_proba = xgb_model2.predict_proba(xgb2_input)[0]
    xgb2_pred = xgb_model2.predict(xgb2_input)[0]

    results['xgb2_pred'] = xgb2_pred
    results['xgb2_home_prob'] = round(xgb2_proba[1], 4)
    results['xgb2_away_prob'] = round(xgb2_proba[0], 4)
    results['xgb2_winner'] = home_name if xgb2_pred == 1 else away_name
    results['xgb2_winner_prob'] = round(xgb2_proba[1] if xgb2_pred == 1 else xgb2_proba[0], 4)

    return results


# ============================================
# ADD TEST CASE FOR MARLON YARBROUGH VS EVAN TALLMADGE
# ============================================

print("\n" + "="*80)
print("🔍 TEST CASE: Marlon Yarbrough vs Evan Tallmadge")
print("="*80)

# Get their data
marlon_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Marlon Yarbrough'].iloc[0]
evan_data = inference_pipeline_db[inference_pipeline_db['wrestler_name'] == 'Evan Tallmadge'].iloc[0]

print(f"\n📊 Wrestler Data:")
print(f"   Marlon Yarbrough - Rank: {marlon_data['rank']}, Team: {marlon_data['team_name']}, Win Rate: {marlon_data['win_rate']}%")
print(f"   Evan Tallmadge   - Rank: {evan_data['rank']}, Team: {evan_data['team_name']}, Win Rate: {evan_data['win_rate']}%")

# Get historical data
hist_features = get_wrestler_match_history('Marlon Yarbrough', 'Evan Tallmadge', matches_reference)
print(f"\n📜 Historical Features: {hist_features}")

# Prepare features
features, home_name, away_name, home, away = prepare_matchup_features(
    'Marlon Yarbrough', 'Evan Tallmadge', hist_features, inference_pipeline_db, weight_class=133
)

print(f"\n🏠 Home: {home_name} (Rank {home['rank']})")
print(f"✈️ Away: {away_name} (Rank {away['rank']})")

# Make prediction
results = predict_single_matchup('Marlon Yarbrough', 'Evan Tallmadge', features, home_name, away_name)

print(f"\n🔮 Prediction Results:")
print(f"   LOGREG: {results['logreg_winner']} wins with {results['logreg_winner_prob']*100:.1f}% confidence")
print(f"   DT    : {results['dt_winner']} wins with {results['dt_winner_prob']*100:.1f}% confidence")
print(f"   XGB1  : {results['xgb1_winner']} wins with {results['xgb1_winner_prob']*100:.1f}% confidence")
print(f"   XGB2  : {results['xgb2_winner']} wins with {results['xgb2_winner_prob']*100:.1f}% confidence")

print("\n" + "="*80)
print("✅ TEST CASE COMPLETE")
print("="*80)


# ============================================
# NOW RUN ALL WEIGHT CLASSES WITH UPDATED FUNCTION
# ============================================

weights = [125, 133, 141, 149, 157, 165, 174, 184, 197, 285]

all_wrestler_list = [
    wrestlers_125, wrestlers_133, wrestlers_141, wrestlers_149,
    wrestlers_157, wrestlers_165, wrestlers_174, wrestlers_184,
    wrestlers_197, wrestlers_285
]

# Dictionary to store all results
all_weight_results = {}

# Loop through each weight class
for weight_idx, (weight, wrestler_list) in enumerate(zip(weights, all_wrestler_list)):
    print("\n" + "="*100)
    print(f"🏋️  PROCESSING {weight}lbs WEIGHT CLASS ({weight_idx+1}/10)")
    print("="*100)

    # Display wrestlers in this weight class
    print(f"\n📋 {weight}lbs Wrestlers:")
    wrestlers_df = inference_pipeline_db[inference_pipeline_db['wrestler_name'].isin(wrestler_list)].copy()
    print(wrestlers_df[['wrestler_name', 'team_name', 'rank', 'win_rate']].to_string(index=False))

    # Generate all matchup predictions for this weight class
    total_matchups = len(list(combinations(wrestler_list, 2)))
    print(f"\n🔄 Generating predictions for {total_matchups} matchups...")

    weight_predictions = []
    matchup_count = 0

    for w1, w2 in combinations(wrestler_list, 2):
        matchup_count += 1
        if matchup_count % 5 == 0 or matchup_count == total_matchups:
            print(f"   Progress: {matchup_count}/{total_matchups} matchups")

        # Get historical data
        hist_features = get_wrestler_match_history(w1, w2, matches_reference)

        # Prepare features
        features, home_name, away_name, home, away = prepare_matchup_features(
            w1, w2, hist_features, inference_pipeline_db, weight_class=weight
        )

        # Make predictions with updated function
        results = predict_single_matchup(w1, w2, features, home_name, away_name)

        # Add weight class
        results['weight_class'] = weight
        weight_predictions.append(results)

    # Create dataframe for this weight class
    weight_df = pd.DataFrame(weight_predictions)

    # Add model agreement column
    weight_df['all_agree'] = (
        (weight_df['logreg_winner'] == weight_df['dt_winner']) &
        (weight_df['dt_winner'] == weight_df['xgb1_winner']) &
        (weight_df['xgb1_winner'] == weight_df['xgb2_winner'])
    )

    # Store results
    all_weight_results[weight] = weight_df

    # ============================================
    # DISPLAY ALL MATCHUPS WITH WINNER PROBABILITIES
    # ============================================
    print(f"\n📊 {weight}lbs - ALL MATCHUP PREDICTIONS")
    print("="*100)

    display_cols = [
        'home_wrestler', 'away_wrestler',
        'logreg_winner', 'logreg_winner_prob',
        'dt_winner', 'dt_winner_prob',
        'xgb1_winner', 'xgb1_winner_prob',
        'xgb2_winner', 'xgb2_winner_prob',
        'all_agree'
    ]

    display_df = weight_df[display_cols].copy()
    display_df.columns = [
        'Home', 'Away',
        'LR_Winner', 'LR_Conf',
        'DT_Winner', 'DT_Conf',
        'X1_Winner', 'X1_Conf',
        'X2_Winner', 'X2_Conf',
        'Agree'
    ]

    # Format confidence as percentage
    for col in ['LR_Conf', 'DT_Conf', 'X1_Conf', 'X2_Conf']:
        display_df[col] = (display_df[col] * 100).round(1).astype(str) + '%'

    print(display_df.to_string(index=False))

    # ============================================
    # QUICK SUMMARY STATS
    # ============================================
    print(f"\n📊 {weight}lbs - QUICK SUMMARY")
    print("-" * 60)

    # Agreement stats
    agreement_count = weight_df['all_agree'].sum()
    print(f"Total Matchups: {len(weight_df)}")
    print(f"All models agree: {agreement_count}/{len(weight_df)} ({agreement_count/len(weight_df)*100:.1f}%)")

    # Calculate average confidence
    weight_df['avg_confidence'] = (
        weight_df['logreg_winner_prob'] +
        weight_df['dt_winner_prob'] +
        weight_df['xgb1_winner_prob'] +
        weight_df['xgb2_winner_prob']
    ) / 4

    # Top 3 most confident predictions
    top_confident = weight_df.nlargest(3, 'avg_confidence')[
        ['home_wrestler', 'away_wrestler', 'avg_confidence']
    ]
    print("\n🔝 Top 3 most confident predictions:")
    for _, row in top_confident.iterrows():
        print(f"   {row['home_wrestler']} vs {row['away_wrestler']}: {row['avg_confidence']*100:.1f}%")

    # Predicted champion by each model
    print("\n🏆 Predicted champions:")
    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        champ_counts = weight_df[f'{model}_winner'].value_counts()
        champ = champ_counts.index[0]
        champ_wins = champ_counts.values[0]
        champ_pct = (champ_wins / (len(wrestler_list) - 1)) * 100
        print(f"   {model.upper():6}: {champ} ({champ_wins}/{len(wrestler_list)-1} wins, {champ_pct:.1f}%)")

    # Check for unanimous champion
    champions = []
    for model in ['logreg', 'dt', 'xgb1', 'xgb2']:
        champions.append(weight_df[f'{model}_winner'].value_counts().index[0])

    if all(c == champions[0] for c in champions):
        print(f"\n✅ UNANIMOUS CHAMPION: {champions[0]}")
    else:
        print(f"\n⚠️ No unanimous champion")
        for model, champ in zip(['logreg', 'dt', 'xgb1', 'xgb2'], champions):
            print(f"   {model.upper():6}: {champ}")

    # Save individual weight class results
    #weight_df.to_csv(f'{weight}lbs_predictions.csv', index=False)
    #print(f"\n💾 Saved {weight}lbs predictions to {weight}lbs_predictions.csv")

    print("\n" + "="*60)

# ============================================
# OVERALL SUMMARY ACROSS ALL WEIGHT CLASSES
# ============================================
print("\n" + "="*100)
print("📊 OVERALL SUMMARY - ALL WEIGHT CLASSES")
print("="*100)

summary_rows = []
for weight in weights:
    weight_df = all_weight_results[weight]

    # Agreement stats
    agreement_count = weight_df['all_agree'].sum()
    agreement_pct = agreement_count / len(weight_df) * 100

    # Champions by model
    logreg_champ = weight_df['logreg_winner'].value_counts().index[0]
    dt_champ = weight_df['dt_winner'].value_counts().index[0]
    xgb1_champ = weight_df['xgb1_winner'].value_counts().index[0]
    xgb2_champ = weight_df['xgb2_winner'].value_counts().index[0]

    # Check unanimous
    unanimous = (logreg_champ == dt_champ == xgb1_champ == xgb2_champ)

    summary_rows.append({
        'Weight': f"{weight}lbs",
        'Matchups': len(weight_df),
        'Agreement': f"{agreement_pct:.1f}%",
        'LogReg Champ': logreg_champ[:15] + "..." if len(logreg_champ) > 15 else logreg_champ,
        'DT Champ': dt_champ[:15] + "..." if len(dt_champ) > 15 else dt_champ,
        'XGB1 Champ': xgb1_champ[:15] + "..." if len(xgb1_champ) > 15 else xgb1_champ,
        'XGB2 Champ': xgb2_champ[:15] + "..." if len(xgb2_champ) > 15 else xgb2_champ,
        'Unanimous': '✅' if unanimous else '❌'
    })

summary_df = pd.DataFrame(summary_rows)
print("\n", summary_df.to_string(index=False))

# ============================================
# SAVE MASTER SUMMARY
# ============================================
#summary_df.to_csv('all_weight_classes_summary.csv', index=False)
print("\n💾 Saved master summary to all_weight_classes_summary.csv")

# Combine all predictions into one master file
all_predictions_master = pd.concat(all_weight_results.values(), ignore_index=True)
all_predictions_master.to_csv('all_predictions_master.csv', index=False)
print("💾 Saved all predictions to all_predictions_master.csv")

print("\n" + "="*100)
print("✅ ALL WEIGHT CLASS PREDICTIONS COMPLETE")
print("="*100)
print(f"Total weight classes processed: {len(weights)}")
print(f"Total matchups processed: {sum(len(df) for df in all_weight_results.values())}")


🔍 TEST CASE: Marlon Yarbrough vs Evan Tallmadge

📊 Wrestler Data:
   Marlon Yarbrough - Rank: 46, Team: Virginia, Win Rate: 25.0%
   Evan Tallmadge   - Rank: 50, Team: Pittsburgh, Win Rate: 57.1%

📜 Historical Features: {'wrestled_before': 0, 'home_point_diff_rematches': 0, 'home_pinned_away': 0, 'total_matches': 0, 'w1_wins': 0, 'w2_wins': 0, 'avg_point_diff_w1': 0, 'last_match_date': None, 'last_winner': None}

🏠 Home: Marlon Yarbrough (Rank 46)
✈️ Away: Evan Tallmadge (Rank 50)

🔮 Prediction Results:
   LOGREG: Evan Tallmadge wins with 66.6% confidence
   DT    : Evan Tallmadge wins with 85.6% confidence
   XGB1  : Evan Tallmadge wins with 79.5% confidence
   XGB2  : Evan Tallmadge wins with 89.3% confidence

✅ TEST CASE COMPLETE

🏋️  PROCESSING 125lbs WEIGHT CLASS (1/10)

📋 125lbs Wrestlers:
     wrestler_name      team_name  rank  win_rate
  Vincent Robinson       NC State     7      80.0
    Tyler Chappell     Pittsburgh    43      35.7
   Eddie Ventresca  Virginia Tech     4   

# Made prediction across all weights

In [ ]:
matches_reference.query('home_name == "Josh Barr" or away_name == "Josh Barr"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
1874,90,197,12/20/2025,764.0,Devin Wasley,50.0,127.0,Josh Barr,1.0,False,LTF,LTF5 19 - 3 3:20,JR,North Dakota State,SO,Penn State
1901,86,197,12/20/2025,263.0,Angelo Posada,20.0,127.0,Josh Barr,1.0,False,LTF,LTF5 19 - 3 4:13,FR,Stanford,SO,Penn State
2829,14,197,1/10/2026,127.0,Josh Barr,1.0,137.0,Remy Cotton,35.0,True,WTF,WTF5 18 - 3 7:00,SO,Penn State,SO,Rutgers
3240,324,197,1/16/2026,1309.0,Brody Sampson,164.0,127.0,Josh Barr,1.0,False,LFALL,LFALL 3:42,RSFR,Iowa,SO,Penn State
3498,309,197,1/18/2026,342.0,Alex Smith,61.0,127.0,Josh Barr,1.0,False,LFALL,LFALL 1:48,FR,Northwestern,SO,Penn State
3587,372,197,1/23/2026,127.0,Josh Barr,1.0,956.0,Gabe Sollars,21.0,True,WMD,WMD 14 - 6,SO,Penn State,SR,Indiana
3807,368,197,1/24/2026,352.0,Branson John,19.0,127.0,Josh Barr,1.0,False,LTF,LTF5 19 - 4 6:51,SO,Maryland,SO,Penn State
3981,411,197,1/30/2026,127.0,Josh Barr,1.0,223.0,Camden McDanel,14.0,True,WMD,WMD 21 - 9,SO,Penn State,SO,Nebraska
4504,513,197,2/6/2026,282.0,Hayden Walters,45.0,127.0,Josh Barr,1.0,False,LTF,LTF5 19 - 4 6:51,RSFR,Michigan,SO,Penn State
5041,467,197,2/13/2026,127.0,Josh Barr,1.0,450.0,Luke Geog,22.0,True,WMD,WMD 11 - 2,SO,Penn State,JR,Ohio State


In [ ]:
matches_reference.query('home_name == "Evan Tallmadge" or away_name == "Evan Tallmadge"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
49,281,133,11/1/2025,837.0,Brendan Ferretti,41.0,295.0,Evan Tallmadge,50.0,False,LSV,LSV-1 6 - 3,SR,Navy,SO,Pittsburgh
291,267,133,11/8/2025,1192.0,Troy Guerra,212.0,295.0,Evan Tallmadge,50.0,False,LTF,LTF5 16 - 1 3:42,JR,Buffalo,SO,Pittsburgh
360,246,133,11/14/2025,295.0,Evan Tallmadge,50.0,1093.0,Ethan Rivera,101.0,True,WDEC,WDEC 7 - 2,SO,Pittsburgh,SO,Princeton
1035,178,133,11/21/2025,196.0,Drake Ayala,6.0,295.0,Evan Tallmadge,50.0,True,WMD,WMD 13 - 4,SR,Iowa,SO,Pittsburgh
1263,151,133,12/5/2025,638.0,Mason Ziegler,100.0,295.0,Evan Tallmadge,50.0,False,LDEC,LDEC 9 - 5,RSFR,Lehigh,SO,Pittsburgh
1440,141,133,12/7/2025,295.0,Evan Tallmadge,50.0,1039.0,Blue Stiffler,112.0,True,WDEC,WDEC 8 - 2,SO,Pittsburgh,FR,Bucknell
1590,122,133,12/13/2025,345.0,Braxton Brown,17.0,295.0,Evan Tallmadge,50.0,True,WDEC,WDEC 4 - 1,SR,Maryland,SO,Pittsburgh
1913,88,133,12/20/2025,295.0,Evan Tallmadge,50.0,768.0,Landon Bainey,184.0,True,WTF,WTF5 19 - 4 5:43,SO,Pittsburgh,RSFR,Edinboro
2582,34,133,1/9/2026,295.0,Evan Tallmadge,50.0,305.0,Dillon Campbell,12.0,False,LDEC,LDEC 5 - 1,SO,Pittsburgh,RSFR,Virginia Tech
3092,330,133,1/16/2026,295.0,Evan Tallmadge,55.0,325.0,Gunner Andrick,27.0,False,LSV,LSV-1 4 - 1,SO,Pittsburgh,FR,West Virginia


In [ ]:
inference_pipeline_db.query('wrestler_name == "Kysen Terukina"')

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff,Class
521,Kysen Terukina,North Carolina,29,29,[125],10,7,3,70.0,1,10.0,0,1,0,6,38.1,4.8,1.7,4.22,SR
